<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/negative_control_fintx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Transactions dataset: a negative control

In `external_dataset_check.ipynb`, every column and every model scored at chance on the Financial Transactions dataset from Kaggle (ROC-AUC about 0.50), so the fraud labels appear to be random. We do not use this dataset for our main results, but we keep it as a **negative control**: a dataset where we expect no signal, which shows that our protocol does not find signal where there is none.

We test this in two ways:
1. **Each column on its own.** We rank the test transactions by every column of the dataset and measure PR-AUC, lift and ROC-AUC.
2. **A label-permutation test.** We train three models (our features, our features with location, and all native columns) twice: once with the real labels and once with the training and validation labels shuffled at random. Shuffling the labels destroys any link between the columns and fraud, so a model trained on shuffled labels can only score at chance. If the models trained on the real labels score the same, the real labels carry no learnable signal either.

The protocol is the same as in `cross_dataset_protocol.ipynb`: split by time with the last 20% of the period as test and the last 20% of the training period as validation, encodings fitted on the training part only, Random Forest Tuned with five seeds, F2 threshold on validation and 500 day-block bootstrap resamples. The test labels are never shuffled.

**Our features on this dataset.** The dataset has no age or gender, so our features are the other four: `amt_log`, `category_risk`, `is_night` and `gap_log` (hours since the same sender account's previous transaction). The location feature is the fraud rate of the transaction's city on the training part (`location_risk`).

**Sample.** The dataset has 5 million transactions from about 900,000 sender accounts. A model trained on random labels grows very deep trees, so to keep the run time reasonable we use a random sample of whole sender accounts with about 500,000 transactions. Keeping whole accounts keeps every account's time gaps complete.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook.

In [1]:
# 1.1 Settings
QUICK    = False         # True = 1 seed, 100 trees, 200 bootstrap resamples (a fast preview only)

SEEDS     = [42] if QUICK else [42, 43, 44, 45, 46]
N_TREES   = 100 if QUICK else 200
N_BOOT    = 200 if QUICK else 500
BOOT_SEED = 2026
PERM_SEED = 7            # the label shuffle for the permutation test
MAX_ROWS  = 500_000      # sample of whole sender accounts
SAMPLE_SEED = 42

KAGGLE_SLUG = 'aryan208/financial-transactions-dataset-for-fraud-detection'
print(f'seeds: {SEEDS} | trees: {N_TREES} | bootstrap: {N_BOOT} | sample: {MAX_ROWS:,} rows')

seeds: [42, 43, 44, 45, 46] | trees: 200 | bootstrap: 500 | sample: 500,000 rows


In [2]:
# 1.2 Download the dataset from Kaggle
import os, glob
from getpass import getpass

DATA_DIR = '/content/data/fintx'
if not glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True):
    os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
    os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')
    !kaggle datasets download -d {KAGGLE_SLUG} -p {DATA_DIR} --unzip -q
for f in sorted(glob.glob(f'{DATA_DIR}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/aryan208/financial-transactions-dataset-for-fraud-detection
License(s): CC0-1.0
    796.1 MB  /content/data/fintx/financial_fraud_detection_dataset.csv


In [3]:
# 1.3 Clone the project repository to use our helpers
REPO = 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'
if not os.path.exists('src/evaluation.py'):
    if not os.path.exists(REPO):
        github_token = getpass('Paste your GitHub token: ').strip()
        !git clone https://{github_token}@github.com/subbareddypullalacheruvu/{REPO}.git
    %cd {REPO}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 381 (delta 156), reused 53 (delta 44), pack-reused 149 (from 1)
Receiving objects: 100% (381/381), 11.99 MiB | 28.35 MiB/s, done.
Resolving deltas: 100% (234/234), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Imports and the Google Drive folders for progress and results
import sys, gc
import numpy as np
from IPython.display import display
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk
from src.evaluation import chronological_val_split, run_seeds, bootstrap_model, summarise, paired_difference, format_ci
pd.set_option('display.max_columns', 60)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol'
CHECKPOINT_DIR = f"{BASE}/fintx{'_quick' if QUICK else ''}"
RESULTS_DIR = f'{BASE}/results'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('progress ->', CHECKPOINT_DIR)
print('results  ->', RESULTS_DIR)

Mounted at /content/drive
progress -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/fintx
results  -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results


## 2. Loading the data
We read the transactions, parse the label and the time, and keep a random sample of whole sender accounts.

In [5]:
# 2.1 Load and sample whole sender accounts
raw = pd.read_csv(max(glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True), key=os.path.getsize), low_memory=False)
print(f'full dataset: {len(raw):,} rows, {raw["sender_account"].nunique():,} sender accounts, '
      f'fraud rate {raw["is_fraud"].astype(str).str.lower().isin(["true", "1"]).mean():.3%}')

accounts = raw['sender_account'].unique()
rng = np.random.default_rng(SAMPLE_SEED)
keep = rng.choice(accounts, size=min(len(accounts), int(len(accounts) * MAX_ROWS / len(raw))), replace=False)
d = raw[raw['sender_account'].isin(set(keep))].copy()
del raw; gc.collect()

d['_time'] = pd.to_datetime(d['timestamp'], errors='coerce')
d['_y'] = d['is_fraud'].astype(str).str.strip().str.lower().isin(['true', '1']).astype(int)
d['_card'] = d['sender_account'].astype(str)
d['_amount'] = pd.to_numeric(d['amount'], errors='coerce')
d['_category'] = d['merchant_category'].astype(str)
d = d.dropna(subset=['_time'])
print(f'sample: {len(d):,} rows, {d["_card"].nunique():,} sender accounts, fraud rate {d["_y"].mean():.3%}')
print(f'{d["_time"].min()} to {d["_time"].max()}')

# identifiers, the time stamp, the label and the column that describes the fraud itself
SKIP = {'transaction_id', 'timestamp', 'sender_account', 'receiver_account', 'is_fraud', 'fraud_type',
        'ip_address', 'device_hash'}
NATIVE_COLS = [c for c in d.columns if c not in SKIP and not c.startswith('_')]
TEXT_COLS = {'transaction_type', 'merchant_category', 'location', 'device_used', 'payment_channel'}
print(f'{len(NATIVE_COLS)} native columns:', NATIVE_COLS)

full dataset: 5,000,000 rows, 896,513 sender accounts, fraud rate 3.591%
sample: 499,768 rows, 89,651 sender accounts, fraud rate 3.589%
2023-01-01 00:21:19.560899 to 2024-01-01 22:50:49.475634
10 native columns: ['amount', 'transaction_type', 'merchant_category', 'location', 'device_used', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'payment_channel']


## 3. Building the features
As in our other notebooks, the time gap only looks back at the same sender account's previous transaction, so we calculate it on the full sample before splitting.

In [6]:
# 3.1 Our features
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
d = d.sort_values(['_card', '_time']).reset_index(drop=True)
d['_gap_h'] = d.groupby('_card')['_time'].diff().dt.total_seconds() / 3600
d['gap_log'] = np.log1p(d['_gap_h']).fillna(-1)
d['amt_log'] = np.log1p(d['_amount'].clip(lower=0)).fillna(0)
d['is_night'] = d['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
d['day'] = d['_time'].dt.floor('D')
d = d.sort_values('_time').reset_index(drop=True)

pd.DataFrame({'fraud': [d.loc[d['_y'] == 1, '_gap_h'].median(), d.loc[d['_y'] == 1, 'is_night'].mean() * 100,
                        d.loc[d['_y'] == 1, '_amount'].median()],
              'normal': [d.loc[d['_y'] == 0, '_gap_h'].median(), d.loc[d['_y'] == 0, 'is_night'].mean() * 100,
                         d.loc[d['_y'] == 0, '_amount'].median()]},
             index=['median time gap (hours)', 'at night (22-03) %', 'median amount'])

,fraud,normal
median time gap (hours),842.227937,889.764781
at night (22-03) %,25.498941,24.954445
median amount,131.295000,138.400000


## 4. Splitting by time and fitting the encodings
The last 20% of the period is the test set, and the last 20% of the training period is validation. `category_risk`, `location_risk` and every text-column encoding are fitted on the training part only.

In [7]:
# 4.1 Train / validation / test and the encodings
cutoff_test = d['_time'].min() + 0.8 * (d['_time'].max() - d['_time'].min())
train, test = d[d['_time'] < cutoff_test], d[d['_time'] >= cutoff_test].copy()
train_sub, val, _ = chronological_val_split(train, '_time', val_frac=0.2)
del train, d; gc.collect()
for name, part in [('train_sub', train_sub), ('val', val), ('test', test)]:
    print(f"{name:9s} {part['_time'].min()} to {part['_time'].max()}  rows={len(part):>9,}  "
          f"fraud={part['_y'].sum():>6,}  rate={part['_y'].mean():.4%}")

parts = (train_sub, val, test)
for col, name in [('_category', 'category_risk'), ('location', 'location_risk')]:
    risk_map = fit_category_risk(train_sub, col, '_y')
    for p in parts:
        p[name] = apply_category_risk(p, col, risk_map)

NATIVE_FEATURES = []
y_fit, prior = train_sub['_y'], train_sub['_y'].mean()
for col in NATIVE_COLS:
    name = f'raw__{col}'
    if col in TEXT_COLS or not pd.api.types.is_numeric_dtype(train_sub[col]):
        rates = y_fit.groupby(train_sub[col].astype(str)).mean()
        for p in parts: p[name] = p[col].astype(str).map(rates).fillna(prior).astype(float)
    else:
        fill = train_sub[col].astype(float).median()
        for p in parts: p[name] = p[col].astype(float).fillna(fill)
    NATIVE_FEATURES.append(name)
y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
PREVALENCE = y_test.mean()
print(f'test fraud rate {PREVALENCE:.4%}; a model that ranks at random scores a PR-AUC of about {PREVALENCE:.4f}')

train_sub 2023-01-01 00:21:19.560899 to 2023-08-23 05:07:30.115165  rows=  320,011  fraud=11,429  rate=3.5714%
val       2023-08-23 05:09:02.598227 to 2023-10-20 18:20:36.111038  rows=   80,176  fraud= 2,936  rate=3.6619%
test      2023-10-20 18:21:52.949298 to 2024-01-01 22:50:49.475634  rows=   99,581  fraud= 3,573  rate=3.5880%
test fraud rate 3.5880%; a model that ranks at random scores a PR-AUC of about 0.0359


## 5. Each column on its own
We rank the test transactions by each column (numeric columns as they are, flipped if lower values mean more fraud on the training part; text columns by the fraud rate of each value on the training part). We include our own features too.

In [8]:
# 5.1 PR-AUC, lift and ROC-AUC of every column on its own
OURS = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'location_risk']
rows = []
for name in NATIVE_FEATURES + OURS:
    x_fit, x_test = train_sub[name], test[name]
    if x_fit.nunique() > 1 and roc_auc_score(y_tr, x_fit) < 0.5:
        x_test = -x_test
    pr = average_precision_score(y_test, x_test)
    rows.append({'column': name.replace('raw__', ''), 'kind': 'ours' if name in OURS else 'native',
                 'PR-AUC': pr, 'lift': pr / PREVALENCE, 'ROC-AUC': roc_auc_score(y_test, x_test)})
single = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
single.round(4)

,column,kind,PR-AUC,lift,ROC-AUC
0,device_used,native,0.0363,1.0129,0.5063
1,velocity_score,native,0.0361,1.0068,0.5054
2,is_night,ours,0.0362,1.0093,0.5046
3,location,native,0.0363,1.0111,0.5029
4,location_risk,ours,0.0363,1.0111,0.5029
5,geo_anomaly_score,native,0.0359,1.0009,0.5022
6,amount,native,0.0363,1.0125,0.5006
7,amt_log,ours,0.0363,1.0125,0.5006
8,category_risk,ours,0.0361,1.0049,0.5005
9,merchant_category,native,0.0361,1.0049,0.5005


## 6. The label-permutation test
We train each model with the real labels and with the training and validation labels shuffled (`PERM_SEED = 7`). The test labels are the real ones in both cases.

In [9]:
# 6.1 Real and shuffled labels
GROUPS = {
    'Our features':            ['amt_log', 'category_risk', 'is_night', 'gap_log'],
    'Our features + location': ['amt_log', 'category_risk', 'is_night', 'gap_log', 'location_risk'],
    'All native columns':      NATIVE_FEATURES,
}
perm = np.random.default_rng(PERM_SEED)
y_tr_shuffled = pd.Series(perm.permutation(y_tr.values), index=y_tr.index)
y_val_shuffled = pd.Series(perm.permutation(y_val.values), index=y_val.index)

def make_rf(n_features):
    return lambda s: RandomForestClassifier(n_estimators=N_TREES, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

runs = {}
for g, cols in GROUPS.items():
    for labels, (a, b) in [('real labels', (y_tr, y_val)), ('shuffled labels', (y_tr_shuffled, y_val_shuffled))]:
        runs[(g, labels)] = run_seeds(f'fintx {g}, {labels}', make_rf(len(cols)),
                                      train_sub[cols], a, val[cols], b, test[cols], SEEDS,
                                      checkpoint_dir=CHECKPOINT_DIR)

fintx Our features, real labels seed 42: loaded from checkpoint
fintx Our features, real labels seed 43: loaded from checkpoint
fintx Our features, real labels seed 44: loaded from checkpoint
fintx Our features, real labels seed 45: loaded from checkpoint
fintx Our features, real labels seed 46: loaded from checkpoint
fintx Our features, shuffled labels seed 42: loaded from checkpoint
fintx Our features, shuffled labels seed 43: loaded from checkpoint
fintx Our features, shuffled labels seed 44: loaded from checkpoint
fintx Our features, shuffled labels seed 45: loaded from checkpoint
fintx Our features, shuffled labels seed 46: loaded from checkpoint
fintx Our features + location, real labels seed 42: loaded from checkpoint
fintx Our features + location, real labels seed 43: loaded from checkpoint
fintx Our features + location, real labels seed 44: loaded from checkpoint
fintx Our features + location, real labels seed 45: loaded from checkpoint
fintx Our features + location, real labe

In [10]:
# 6.2 Bootstrap, and real labels against shuffled labels
test_days = test['day'].values
boots, rows = {}, []
for (g, labels), seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boots[(g, labels)] = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    s = summarise(y_test, pairs, boots[(g, labels)])
    rows.append({'Model': g, 'Labels': labels, 'Prevalence': PREVALENCE,
                 'PR-AUC': s['PR-AUC'], 'PR-AUC CI low': s['PR-AUC CI low'], 'PR-AUC CI high': s['PR-AUC CI high'],
                 'Lift': s['PR-AUC'] / PREVALENCE,
                 'Lift CI low': s['PR-AUC CI low'] / PREVALENCE, 'Lift CI high': s['PR-AUC CI high'] / PREVALENCE,
                 'ROC-AUC': np.mean([roc_auc_score(y_test, r['test_proba']) for r in seed_runs]),
                 'F2': s['F2'], 'MCC': s['MCC'], 'MCC CI low': s['MCC CI low'], 'MCC CI high': s['MCC CI high']})
    print(f'{g}, {labels}: done')
result = pd.DataFrame(rows)
for g in GROUPS:
    dd = paired_difference(boots[(g, 'real labels')], boots[(g, 'shuffled labels')], 'PR-AUC')
    i = result.index[(result['Model'] == g) & (result['Labels'] == 'real labels')][0]
    result.loc[i, ['Real minus shuffled', 'Diff CI low', 'Diff CI high']] = [dd['mean diff'], dd['CI low'], dd['CI high']]

out = f"{RESULTS_DIR}/point7_fintx_negative_control{'_quick' if QUICK else ''}.csv"
result.to_csv(out, index=False)
os.makedirs('results', exist_ok=True)
result.to_csv(f"results/point7_fintx_negative_control{'_quick' if QUICK else ''}.csv", index=False)
print('saved ->', out)
print(f'test fraud rate {PREVALENCE:.4%} ({int(y_test.sum()):,} fraud of {len(y_test):,})')

pd.DataFrame({
    'PR-AUC [95% CI]': [f"{r['PR-AUC']:.4f} [{r['PR-AUC CI low']:.4f}, {r['PR-AUC CI high']:.4f}]" for _, r in result.iterrows()],
    'Lift [95% CI]': [f"{r['Lift']:.2f} [{r['Lift CI low']:.2f}, {r['Lift CI high']:.2f}]" for _, r in result.iterrows()],
    'ROC-AUC': result['ROC-AUC'].round(4).values,
    'MCC [95% CI]': [f"{r['MCC']:.4f} [{r['MCC CI low']:.4f}, {r['MCC CI high']:.4f}]" for _, r in result.iterrows()],
    'Real minus shuffled [95% CI]': ['' if pd.isna(r.get('Real minus shuffled')) else
                                     f"{r['Real minus shuffled']:.4f} [{r['Diff CI low']:.4f}, {r['Diff CI high']:.4f}]"
                                     for _, r in result.iterrows()],
}, index=pd.MultiIndex.from_frame(result[['Model', 'Labels']]))

Our features, real labels: done
Our features, shuffled labels: done
Our features + location, real labels: done
Our features + location, shuffled labels: done
All native columns, real labels: done
All native columns, shuffled labels: done
saved -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results/point7_fintx_negative_control.csv
test fraud rate 3.5880% (3,573 fraud of 99,581)


PR-AUC [95% CI]  \
Model                   Labels                                     
Our features            real labels      0.0360 [0.0345, 0.0378]   
                        shuffled labels  0.0363 [0.0347, 0.0382]   
Our features + location real labels      0.0358 [0.0344, 0.0375]   
                        shuffled labels  0.0358 [0.0342, 0.0378]   
All native columns      real labels      0.0420 [0.0402, 0.0440]   
                        shuffled labels  0.0393 [0.0374, 0.0412]   

                                             Lift [95% CI]  ROC-AUC  \
Model                   Labels                                        
Our features            real labels      1.00 [0.96, 1.05]   0.5031   
                        shuffled labels  1.01 [0.97, 1.07]   0.5035   
Our features + location real labels      1.00 [0.96, 1.04]   0.4990   
                        shuffled labels  1.00 [0.95, 1.05]   0.4993   
All native columns      real labels      1.17 [1.12, 1.22]   0.5829   
                        shuffled labels  1.10 [1.04, 1.15]   0.5358   

                                                     MCC [95% CI]  \
Model                   Labels                                      
Our features            real labels      0.0051 [-0.0012, 0.0112]   
                        shuffled labels  0.0029 [-0.0024, 0.0084]   
Our features + location real labels      0.0002 [-0.0037, 0.0038]   
                        shuffled labels  0.0014 [-0.0027, 0.0065]   
All native columns      real labels       0.0873 [0.0851, 0.0895]   
                        shuffled labels   0.0036 [0.0012, 0.0055]   

                                        Real minus shuffled [95% CI]  
Model                   Labels                                        
Our features            real labels        -0.0003 [-0.0019, 0.0012]  
                        shuffled labels                               
Our features + location real labels        -0.0001 [-0.0015, 0.0015]  
                        shuffled labels                               
All native columns      real labels          0.0027 [0.0015, 0.0038]  
                        shuffled labels